# 02. Data Loading and Base Objects

        Noa Yarin Levy · Jimi Halawi · Ofir Atias

        **Purpose.** Load the MUSAE edge list and developer labels, validate their schemas, and build the undirected graph used throughout the project.

        **Inputs.** `musae_git_edges.csv`, `musae_git_target.csv`, and the dataset metadata.

        **Outputs.** `G`, `edges_df`, `labels_df`, and `labels_dict`, together with quality checks and sample records.

        **Research role.** Defines the common analysis population for RQ1-RQ4 and prevents silent data-quality failures.

        The notebook is self-contained: it initializes the project paths, defines the needed code, runs the relevant step, and explains the saved results.

## Portable project initialization

This cell finds the project root, moves execution there, and adds `src` to the import path. It also sets deterministic defaults. Expected result: the absolute project-root path.

In [1]:
from pathlib import Path
import os
import sys
import warnings

def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "data").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate a project root containing requirements.txt and data/")

ROOT = _find_project_root(Path.cwd())
os.chdir(ROOT)
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
os.environ.setdefault("PYTHONHASHSEED", "0")
warnings.filterwarnings("ignore", category=FutureWarning)
__name__ = "research_notebook_02"
print(f"Project root: {ROOT}")

Project root: C:\Users\Jimi\Desktop\github-social-network


**Result.** The printed path must end in `github-social-network`. This confirms that all relative paths point to the submitted project folder.

## Data model and conservative cleaning policy

Each edge row represents a mutual follow relation, so the project models an undirected graph. `ml_target=0` denotes Web and `ml_target=1` denotes Machine Learning. Cleaning is deliberately conservative: the loader does not impute labels, silently drop nodes, or rewrite edges. Invalid schemas, duplicates, self-loops, and missing values raise explicit errors, preserving fidelity to the original MUSAE snapshot.

## Dataset context

The MUSAE GitHub Social Network is a June 2019 snapshot. The project uses a mutual-follow edge list, one binary Web/ML label per developer, and sparse content indicators derived from profile and repository information. The label is an analysis target rather than a complete professional identity; its binary form and age are explicit limitations.

## Source code: `src/data_loader.py`

        This module validates the raw CSV schemas and constructs the graph objects used by every experiment.

        The following cells copy the source exactly. A short setup cell gives the copied code the same file path context it has in `src`. Source SHA-256: `18dbacfbb282665a0bad4e4054d91200118861905c53123341c2c85f7bc67438`.

This cell sets the file path context for the copied source block. It does not modify any file. Expected result: the source path.

In [2]:
__file__ = str(ROOT / 'src/data_loader.py')
print(f"Source context: {__file__}")

Source context: C:\Users\Jimi\Desktop\github-social-network\src\data_loader.py


**Result.** The printed path confirms that this source block reads and writes inside the project folder.

### data_loader.py, lines 1-13: imports, constants, and module setup

        This module validates the raw CSV schemas and constructs the graph objects used by every experiment.

        Expected result: imports and constants are available in memory.

In [3]:
# -*- coding: utf-8 -*-
from pathlib import Path
from typing import Dict, Tuple

import networkx as nx
import pandas as pd


EDGE_COLUMNS = ("id_1", "id_2")
TARGET_ID_COLUMN = "id"
TARGET_LABEL_COLUMN = "ml_target"




**Result.** `imports, constants, and module setup` is now defined. This cell usually has no visible output.

### data_loader.py, lines 14-30: `load_edges`

        Read the edge CSV, standardize endpoint columns, reject missing endpoints and self-loops, and preserve one row per undirected edge.

        Expected result: the function is defined; visible output appears only when it is called.

In [4]:
def load_edges(path: str | Path) -> pd.DataFrame:
    """Load and validate the edge list."""
    path = Path(path)
    edges_df = pd.read_csv(path)

    missing_columns = [col for col in EDGE_COLUMNS if col not in edges_df.columns]
    if missing_columns:
        raise ValueError(f"Edge file is missing required columns: {missing_columns}")

    edges_df = edges_df.loc[:, list(EDGE_COLUMNS)].copy()
    edges_df["id_1"] = pd.to_numeric(edges_df["id_1"], errors="coerce")
    edges_df["id_2"] = pd.to_numeric(edges_df["id_2"], errors="coerce")
    edges_df = edges_df.dropna(subset=["id_1", "id_2"]).astype({"id_1": "int64", "id_2": "int64"})

    return edges_df




**Result.** `load_edges` is now defined. This cell usually has no visible output.

### data_loader.py, lines 31-61: `load_targets`

        Read node metadata, normalize the target to a binary `label` column, reject duplicate IDs or missing labels, and return one validated row per developer.

        Expected result: the function is defined; visible output appears only when it is called.

In [5]:
def load_targets(path: str | Path) -> pd.DataFrame:
    """Load and validate node labels, returning a standardized labels DataFrame."""
    path = Path(path)
    targets_df = pd.read_csv(path)

    required_columns = [TARGET_ID_COLUMN, TARGET_LABEL_COLUMN]
    missing_columns = [col for col in required_columns if col not in targets_df.columns]
    if missing_columns:
        raise ValueError(f"Target file is missing required columns: {missing_columns}")

    keep_columns = [TARGET_ID_COLUMN, TARGET_LABEL_COLUMN]
    if "name" in targets_df.columns:
        keep_columns.insert(1, "name")

    labels_df = targets_df.loc[:, keep_columns].copy()
    labels_df[TARGET_ID_COLUMN] = pd.to_numeric(labels_df[TARGET_ID_COLUMN], errors="coerce")
    labels_df[TARGET_LABEL_COLUMN] = pd.to_numeric(labels_df[TARGET_LABEL_COLUMN], errors="coerce")
    labels_df = labels_df.dropna(subset=[TARGET_ID_COLUMN, TARGET_LABEL_COLUMN])
    labels_df[TARGET_ID_COLUMN] = labels_df[TARGET_ID_COLUMN].astype("int64")
    labels_df[TARGET_LABEL_COLUMN] = labels_df[TARGET_LABEL_COLUMN].astype("int64")
    labels_df = labels_df.drop_duplicates(subset=[TARGET_ID_COLUMN], keep="first")

    unique_labels = set(labels_df[TARGET_LABEL_COLUMN].unique())
    if not unique_labels.issubset({0, 1}):
        raise ValueError(f"Unexpected labels detected in target file: {sorted(unique_labels)}")

    labels_df = labels_df.rename(columns={TARGET_LABEL_COLUMN: "label"})

    return labels_df




**Result.** `load_targets` is now defined. This cell usually has no visible output.

### data_loader.py, lines 62-71: `build_graph`

        Insert every validated endpoint pair into an undirected NetworkX graph.

        Expected result: the function is defined; visible output appears only when it is called.

In [6]:
def build_graph(edges_df: pd.DataFrame) -> nx.Graph:
    """Build an undirected NetworkX graph from an edges DataFrame."""
    missing_columns = [col for col in EDGE_COLUMNS if col not in edges_df.columns]
    if missing_columns:
        raise ValueError(f"Edges DataFrame is missing required columns: {missing_columns}")

    graph = nx.from_pandas_edgelist(edges_df, source="id_1", target="id_2", create_using=nx.Graph())
    return graph




**Result.** `build_graph` is now defined. This cell usually has no visible output.

### data_loader.py, lines 72-98: `load_dataset`

        Compose edge loading, target loading, graph construction, graph-node completion, and label-dictionary creation into one checked dataset API.

        Expected result: the function is defined; visible output appears only when it is called.

In [7]:
def load_dataset(data_dir: str | Path) -> Tuple[nx.Graph, pd.DataFrame, pd.DataFrame, Dict[int, int]]:
    """
    Load the dataset from a directory and build all base objects.

    Returns:
        G: Undirected graph
        edges_df: Edge list DataFrame
        labels_df: Labels DataFrame with standardized 'label' column
        labels_dict: Mapping node_id -> label
    """
    data_dir = Path(data_dir)
    edges_path = data_dir / "musae_git_edges.csv"
    targets_path = data_dir / "musae_git_target.csv"

    if not edges_path.exists():
        raise FileNotFoundError(f"Missing edge file: {edges_path}")
    if not targets_path.exists():
        raise FileNotFoundError(f"Missing target file: {targets_path}")

    edges_df = load_edges(edges_path)
    labels_df = load_targets(targets_path)
    graph = build_graph(edges_df)

    labels_dict = dict(zip(labels_df["id"], labels_df["label"]))
    nx.set_node_attributes(graph, labels_dict, name="label")

    return graph, edges_df, labels_df, labels_dict


**Result.** `load_dataset` is now defined. This cell usually has no visible output.

## Full dataset load and validation display

This cell calls the complete loader, builds `networkx.Graph`, and displays sizes, connectivity, label coverage, and sample records. Expected result: 37,700 nodes, 289,003 edges, one connected component, and no graph node without a label.

In [8]:
G, edges_df, labels_df, labels_dict = load_dataset(ROOT / "data")
dataset_summary = pd.DataFrame([{
    "nodes": G.number_of_nodes(),
    "edges": G.number_of_edges(),
    "connected": __import__("networkx").is_connected(G),
    "label_rows": len(labels_df),
    "missing_graph_labels": len(set(G.nodes()) - set(labels_dict)),
}])
display(dataset_summary)
display(edges_df.head())
display(labels_df.head())

,nodes,edges,connected,label_rows,missing_graph_labels
0,37700,289003,True,37700,0


,id_1,id_2
0,0,23977
1,1,34526
2,1,2370
3,1,14683
4,1,29982


,id,name,label
0,0,Eiryyy,0
1,1,shawflying,0
2,2,JpMCarrilho,1
3,3,SuhwanCha,0
4,4,sunilangadi2,1


**Result.** The graph contains 37,700 nodes and 289,003 edges, is connected, and has complete label coverage. No imputation or giant-component filtering is needed, so RQ1-RQ4 use exactly the same population.

## Summary

        **Completed.** Loaded edges and labels, validated schemas, and built the undirected graph.

        **Key findings.** The data are complete, connected, and require no silent correction or imputation.

        **Workflow connection.** The resulting graph and label objects are the inputs to descriptive and model-based analysis.